# 01 — Gaussian + Zernike Fit

Synthesise a realistic beam (Gaussian main lobe with analytic Zernike perturbations),  
then run `GaussianFit` → `ZernikeFit` on it and inspect the residuals.

**No external data needed** — the beam cube is built entirely from `numpy`.

In [ ]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from beamlab21.fitting import GaussianFit, ZernikeFit
from beamlab21.models import twoD_Gaussian
from beamlab21.zernike import zernike_mode

## 1 · Synthesise a beam cube

Truth: a 2D Gaussian at 400 MHz plus a small coma (Zernike Z₇) perturbation.

In [ ]:
# Grid
N_pix = 201
xy_max = 30.0  # degrees
x = np.linspace(-xy_max, xy_max, N_pix)
y = np.linspace(-xy_max, xy_max, N_pix)
X, Y = np.meshgrid(x, y)

# True Gaussian parameters: [amp, sigx, sigy, xo, yo, tilt_deg]
TRUE_GAUSS = [1.0, 8.0, 7.0, 0.3, -0.2, 5.0]
beam_gauss = twoD_Gaussian(x, y, TRUE_GAUSS)

# Add a Zernike perturbation: coma Z7 (n=3, m=1)
sigx, sigy = TRUE_GAUSS[1], TRUE_GAUSS[2]
xm = X / sigx
ym = Y / sigy
rm = np.hypot(xm, ym)
rm[rm == 0] = 1e-10
tm = np.arctan2(ym, xm)
z7 = zernike_mode(3, 1, rm, tm)        # coma
z4 = zernike_mode(2, 0, rm, tm)        # defocus

COMA_AMP   = 0.04
DEFOCUS_AMP = 0.03
beam_synth = beam_gauss + COMA_AMP * z7 + DEFOCUS_AMP * z4
beam_synth = np.clip(beam_synth / beam_synth.max(), 0, None)  # normalise

# Add a tiny noise floor
rng = np.random.default_rng(42)
beam_synth += rng.normal(0, 5e-4, beam_synth.shape)
beam_synth = np.clip(beam_synth, 0, None)

freq_mhz   = 400.0
freq_arr   = np.array([freq_mhz / 1e3])   # GHz
data_cube  = beam_synth[np.newaxis, :, :]

print(f"Beam cube shape : {data_cube.shape}")
print(f"Peak value      : {data_cube.max():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.pcolormesh(x, y, beam_synth, cmap="inferno", vmin=0)
plt.colorbar(im, ax=ax, label="Normalised amplitude")
ax.set_xlabel("x [deg]"); ax.set_ylabel("y [deg]")
ax.set_title("Synthetic beam (Gaussian + coma + defocus)")
plt.tight_layout()
plt.show()

## 2 · Save to a temporary `.npz` and run `GaussianFit`

In [ ]:
# Write to a temp file so GaussianFit can load it via the standard path
_tmp = tempfile.NamedTemporaryFile(suffix=".npz", delete=False)
_tmp.close()
tmpfile = _tmp.name
np.savez(tmpfile, x=x, y=y, freq=freq_arr, data=data_cube)

# Initial guess: amplitude near 1, sigmas ~ half the HPBW, centre at origin
init_gauss = [0.95, 9.0, 9.0, 0.0, 0.0, 0.0]

gfit = GaussianFit(tmpfile, freq=freq_mhz, normalize_data=True)
out  = gfit.optimize_Gauss(init_gauss)

x_ax, y_ax, xo, yo, fa, freq, sigx_g, sigy_g, gauss_model, data_slice, chi = out
print(f"\nFitted Gaussian  →  sigx={sigx_g:.3f}°  sigy={sigy_g:.3f}°  "
      f"xo={xo:.3f}°  yo={yo:.3f}°  χ²={chi:.4f}")
print(f"Truth            →  sigx={TRUE_GAUSS[1]}°  sigy={TRUE_GAUSS[2]}°  "
      f"xo={TRUE_GAUSS[3]}°  yo={TRUE_GAUSS[4]}°")

## 3 · Run `ZernikeFit` (fast, skip scale-param optimisation)

In [ ]:
N_modes = 15   # number of Zernike modes
fac     = 2.5  # scale-param shrink factor (sigx_zt = sigx_gauss / fac)

zfit = ZernikeFit(
    x_ax, y_ax, xo, yo,
    freq_arr=fa, freq=freq,
    data=data_slice,
    N=N_modes,
    error_type="proportional",
    normalize_data=True,
)
sigx_zt, sigy_zt, coef, model_zt = zfit.NO_optimize_ZT([sigx_g, sigy_g], fac=fac)
print(f"\nZernike scale params  →  sigx={sigx_zt:.3f}°  sigy={sigy_zt:.3f}°")
print(f"Number of coefficients: {len(coef)}")

## 4 · Visualise: data · Gaussian · Zernike · residual

In [ ]:
data_norm = data_slice / data_slice.max()
resid_g   = data_norm - gauss_model / gauss_model.max()
resid_zt  = data_norm - model_zt / model_zt.max()

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
kw  = dict(cmap="inferno", vmin=0, vmax=1)
rkw = dict(cmap="RdBu_r", vmin=-0.05, vmax=0.05)

axes[0].pcolormesh(x, y, data_norm, **kw)
axes[0].set_title("Data (normalised)")

axes[1].pcolormesh(x, y, gauss_model / gauss_model.max(), **kw)
axes[1].set_title("Gaussian model")

axes[2].pcolormesh(x, y, model_zt / model_zt.max(), **kw)
axes[2].set_title("Zernike model")

im = axes[3].pcolormesh(x, y, resid_zt, **rkw)
plt.colorbar(im, ax=axes[3], label="residual")
axes[3].set_title("Residual (data − Zernike)")

for ax in axes:
    ax.set_xlabel("x [deg]")
    ax.set_ylabel("y [deg]")
plt.tight_layout()
plt.show()

Path(tmpfile).unlink(missing_ok=True)

## 5 · Inspect Zernike coefficients

In [ ]:
from beamlab21.zernike import NollToQuantum

j_idx = np.arange(1, len(coef) + 1)
labels = [f"Z{j}\n(n={NollToQuantum(j-1)[0]},m={NollToQuantum(j-1)[1]})" for j in j_idx]

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(j_idx, coef, color="steelblue", edgecolor="k", linewidth=0.5)
ax.set_xticks(j_idx)
ax.set_xticklabels(labels, fontsize=7)
ax.axhline(0, color="k", linewidth=0.8)
ax.set_xlabel("Noll index j")
ax.set_ylabel("Coefficient")
ax.set_title(f"Zernike coefficients  (N={N_modes} modes, fac={fac})")
plt.tight_layout()
plt.show()